# Dashboard Économique Afrique
**Source** : Banque mondiale (WDI)  
**Période** : 2004 – 2023 | **Pays** : 54 pays africains

## 1. Imports & Constantes

In [11]:
from datetime import date
from functools import reduce
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ANNEE_DEBUT = 2004
ANNEE_FIN   = 2023
URL_BASE    = "https://api.worldbank.org/v2"
SEUIL_NAN   = 4

DOSSIER_DATA  = Path("data")
DOSSIER_DATA.mkdir(exist_ok=True)
FICHIER_BRUT   = DOSSIER_DATA / "dataset_economique_afrique_brut.csv"
FICHIER_PROPRE = DOSSIER_DATA / "dataset_economique_afrique_propre.csv"

AFRIQUE = [
    "DZA", "AGO", "BEN", "BWA", "BFA", "BDI", "CPV",
    "CMR", "CAF", "TCD", "COM", "COG", "COD", "CIV",
    "DJI", "EGY", "GNQ", "ERI", "SWZ", "ETH", "GAB",
    "GMB", "GHA", "GIN", "GNB", "KEN", "LSO", "LBR",
    "LBY", "MDG", "MWI", "MLI", "MRT", "MUS", "MAR",
    "MOZ", "NAM", "NER", "NGA", "RWA", "STP", "SEN",
    "SYC", "SLE", "SOM", "ZAF", "SSD", "SDN", "TZA",
    "TGO", "TUN", "UGA", "ZMB", "ZWE",
]

INDICATEURS = {
    "population"               : "SP.POP.TOTL",
    "pib_usd"                  : "NY.GDP.MKTP.CD",
    "pib_par_habitant_usd"     : "NY.GDP.PCAP.CD",
    "croissance_pib_pct"       : "NY.GDP.MKTP.KD.ZG",
    "inflation_pct"            : "FP.CPI.TOTL.ZG",
    "esperance_vie"            : "SP.DYN.LE00.IN",
    "acces_electricite_pct"    : "EG.ELC.ACCS.ZS",
    "utilisateurs_internet_pct": "IT.NET.USER.ZS",
}

SOUS_REGIONS = {
    "Afrique du Nord"   : ["DZA", "EGY", "LBY", "MAR", "TUN", "SDN"],
    "Afrique de l'Ouest": ["BEN", "BFA", "CPV", "CIV", "GMB", "GHA", "GIN", "GNB",
                            "LBR", "MLI", "MRT", "NER", "NGA", "SEN", "SLE", "TGO"],
    "Afrique centrale"  : ["AGO", "CMR", "CAF", "TCD", "COG", "COD", "GNQ", "GAB", "STP"],
    "Afrique de l'Est"  : ["BDI", "COM", "DJI", "ERI", "ETH", "KEN", "MDG", "MWI", "MUS",
                            "MOZ", "RWA", "SYC", "SOM", "SSD", "UGA", "TZA", "ZMB", "ZWE"],
    "Afrique australe"  : ["BWA", "SWZ", "LSO", "NAM", "ZAF"],
}

NOMS_FR = {
    "DZA": "Algérie",       "AGO": "Angola",          "BEN": "Bénin",
    "BWA": "Botswana",       "BFA": "Burkina Faso",    "BDI": "Burundi",
    "CPV": "Cap-Vert",       "CMR": "Cameroun",        "CAF": "République centrafricaine",
    "TCD": "Tchad",          "COM": "Comores",         "COG": "Congo",
    "COD": "RD Congo",       "CIV": "Côte d'Ivoire",   "DJI": "Djibouti",
    "EGY": "Égypte",         "GNQ": "Guinée équatoriale", "ERI": "Érythrée",
    "SWZ": "Eswatini",       "ETH": "Éthiopie",        "GAB": "Gabon",
    "GMB": "Gambie",         "GHA": "Ghana",           "GIN": "Guinée",
    "GNB": "Guinée-Bissau",  "KEN": "Kenya",           "LSO": "Lesotho",
    "LBR": "Libéria",        "LBY": "Libye",           "MDG": "Madagascar",
    "MWI": "Malawi",         "MLI": "Mali",            "MRT": "Mauritanie",
    "MUS": "Maurice",        "MAR": "Maroc",           "MOZ": "Mozambique",
    "NAM": "Namibie",        "NER": "Niger",           "NGA": "Nigéria",
    "RWA": "Rwanda",         "STP": "Sao Tomé-et-Principe", "SEN": "Sénégal",
    "SYC": "Seychelles",     "SLE": "Sierra Leone",    "SOM": "Somalie",
    "ZAF": "Afrique du Sud", "SSD": "Soudan du Sud",   "SDN": "Soudan",
    "TZA": "Tanzanie",       "TGO": "Togo",            "TUN": "Tunisie",
    "UGA": "Ouganda",        "ZMB": "Zambie",          "ZWE": "Zimbabwe",
}

## 2. Téléchargement & Sauvegarde (fichier brut)

In [12]:
def recuperer_api(url, parametres):
    reponse = requests.get(url, params=parametres, timeout=120)
    reponse.raise_for_status()
    return reponse.json()


def verifier_reponse(resultat, nom):
    if not resultat or len(resultat) < 2 or resultat[1] is None:
        raise ValueError(f"Aucune donnée reçue pour {nom}")
    if resultat[0].get("pages", 1) > 1:
        raise ValueError(f"{nom} : la réponse contient plusieurs pages")


def telecharger_infos_pays():
    resultat = recuperer_api(f"{URL_BASE}/country", {"format": "json", "per_page": 400})
    verifier_reponse(resultat, "liste des pays")
    lignes = []
    for p in resultat[1]:
        if p["id"] in AFRIQUE:
            lignes.append({
                "iso3"                  : p["id"],
                "pays_en"               : p["name"],
                "region_banque_mondiale": p.get("region", {}).get("value"),
                "groupe_revenu"         : p.get("incomeLevel", {}).get("value"),
                "latitude"              : p.get("latitude"),
                "longitude"             : p.get("longitude"),
            })
    infos = pd.DataFrame(lignes)
    infos["latitude"]  = pd.to_numeric(infos["latitude"],  errors="coerce")
    infos["longitude"] = pd.to_numeric(infos["longitude"], errors="coerce")
    if len(infos) != len(AFRIQUE):
        raise ValueError(f"{len(infos)} pays reconnus au lieu de {len(AFRIQUE)}")
    return infos


def telecharger_indicateur(nom, code):
    codes_pays = ";".join(AFRIQUE)
    url = f"{URL_BASE}/country/{codes_pays}/indicator/{code}"
    parametres = {"format": "json", "date": f"{ANNEE_DEBUT}:{ANNEE_FIN}", "per_page": 20000}
    resultat = recuperer_api(url, parametres)
    verifier_reponse(resultat, nom)
    df = pd.json_normalize(resultat[1])
    df = df.rename(columns={"countryiso3code": "iso3", "date": "annee", "value": nom})
    df = df[["iso3", "annee", nom]]
    df["annee"] = pd.to_numeric(df["annee"], errors="coerce").dropna().astype(int)
    df[nom]     = pd.to_numeric(df[nom], errors="coerce")
    return df[df["iso3"].isin(AFRIQUE)]


# --- Exécution ---
infos = telecharger_infos_pays()
print("Pays reconnus :", len(infos))

tables = []
for nom, code in INDICATEURS.items():
    table = telecharger_indicateur(nom, code)
    print(f"Téléchargé : {nom} ({len(table)} observations)")
    tables.append(table)

donnees = reduce(
    lambda g, d: pd.merge(g, d, on=["iso3", "annee"], how="outer", validate="one_to_one"),
    tables,
)
donnees = donnees.merge(infos, on="iso3", how="left", validate="many_to_one")

colonnes = [
    "iso3", "pays_en", "annee",
    "region_banque_mondiale", "groupe_revenu", "latitude", "longitude",
] + list(INDICATEURS)
donnees = donnees[colonnes].sort_values(["pays_en", "annee"]).reset_index(drop=True)

donnees.to_csv(FICHIER_BRUT, index=False, encoding="utf-8-sig")
print(f"\nFichier brut sauvegardé : {FICHIER_BRUT}")
print(f"Lignes : {len(donnees)} | Colonnes : {len(donnees.columns)}")
print(f"Période : {donnees['annee'].min()} – {donnees['annee'].max()}")
print(f"Source : Banque mondiale (WDI), téléchargé le {date.today().strftime('%d/%m/%Y')}")

Pays reconnus : 54
Téléchargé : population (1080 observations)
Téléchargé : pib_usd (1080 observations)
Téléchargé : pib_par_habitant_usd (1080 observations)
Téléchargé : croissance_pib_pct (1080 observations)
Téléchargé : inflation_pct (1080 observations)
Téléchargé : esperance_vie (1080 observations)
Téléchargé : acces_electricite_pct (1080 observations)
Téléchargé : utilisateurs_internet_pct (1080 observations)

Fichier brut sauvegardé : data\dataset_economique_afrique_brut.csv
Lignes : 1080 | Colonnes : 15
Période : 2004 – 2023
Source : Banque mondiale (WDI), téléchargé le 27/10/2026


## 3. Exploration des valeurs manquantes

In [13]:
df = pd.read_csv(FICHIER_BRUT)
print("=== Info général ===")
df.info()

print("\n=== NaN par colonne ===")
print(df.isna().sum())

print("\n=== NaN par pays (colonnes indicateurs) ===")
cols_ind = list(INDICATEURS)
na_pays = df[cols_ind].isna().groupby(df["pays_en"]).sum()
print(na_pays[na_pays.sum(axis=1) > 0])

print("\n=== Sauts anormaux d'espérance de vie (> 5 ans) ===")
saut = df.groupby("iso3")["esperance_vie"].diff().abs()
print(df.loc[saut > 5, ["pays_en", "annee", "esperance_vie"]].to_string())

=== Info général ===
<class 'pandas.DataFrame'>
RangeIndex: 1080 entries, 0 to 1079
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   iso3                       1080 non-null   str    
 1   pays_en                    1080 non-null   str    
 2   annee                      1080 non-null   int64  
 3   region_banque_mondiale     1080 non-null   str    
 4   groupe_revenu              1080 non-null   str    
 5   latitude                   1080 non-null   float64
 6   longitude                  1080 non-null   float64
 7   population                 1080 non-null   int64  
 8   pib_usd                    1056 non-null   float64
 9   pib_par_habitant_usd       1056 non-null   float64
 10  croissance_pib_pct         1045 non-null   float64
 11  inflation_pct              1011 non-null   float64
 12  esperance_vie              1080 non-null   float64
 13  acces_electricite_pct      1074 non-nu

## 4. Nettoyage des données

In [14]:
df = pd.read_csv(FICHIER_BRUT)

# --- 4.1 Noms français & sous-région ---
df["pays"]        = df["iso3"].map(NOMS_FR)
df["sous_region"] = df["iso3"].map(
    {iso: reg for reg, codes in SOUS_REGIONS.items() for iso in codes}
)
print("Pays sans nom FR :",      df["pays"].isna().sum())
print("Pays sans sous-région :", df["sous_region"].isna().sum())

# --- 4.2 Fonction d'imputation ---
def combler(df, col, methode):
    nb_na    = df.groupby("pays")[col].transform(lambda s: s.isna().sum())
    eligible = nb_na.between(1, SEUIL_NAN)
    if methode == "interpolation":
        remplie = df.groupby("pays")[col].transform(
            lambda s: s.interpolate(limit_direction="both"))
    else:
        remplie = df.groupby("pays")[col].transform(lambda s: s.fillna(s.median()))
    df.loc[eligible, col] = remplie[eligible]

for col in ["utilisateurs_internet_pct", "acces_electricite_pct"]:
    combler(df, col, "interpolation")
combler(df, "inflation_pct", "mediane")

print("\n=== NaN restants après imputation ===")
print(df.isna().sum())
print("\nNote : les NaN restants concernent des pays avec données structurellement"
      " absentes (Érythrée, Soudan du Sud…) — conservés tels quels.")

# --- 4.3 Correction ciblée RCA : valeurs aberrantes d'espérance de vie ---
masque_rca = (df["iso3"] == "CAF") & (df["esperance_vie"] < 42)
print(df.loc[masque_rca, ["pays", "annee", "esperance_vie"]].to_string(index=False))
df.loc[masque_rca, "esperance_vie"] = float("nan")
print("Valeurs mises en NaN :", masque_rca.sum())
print("Nouveau minimum :", round(df["esperance_vie"].min(), 1))
combler(df, "esperance_vie", "interpolation")

# --- 4.4 Détection & écrêtage des outliers ---
def bornes_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["croissance_pib_pct", "inflation_pct"]:
    bas, haut = bornes_iqr(df[col].dropna())
    df[f"{col}_outlier"] = (df[col] < bas) | (df[col] > haut)
    bas_p, haut_p = df[col].quantile([0.05, 0.95])
    df[f"{col}_ecrete"] = df[col].clip(lower=bas_p, upper=haut_p)
    n = df[f"{col}_outlier"].sum()
    print(f"\n{col} : bornes IQR [{bas:.1f} ; {haut:.1f}] → {n} outliers détectés")

# --- 4.5 Colonnes dérivées ---
df["pib_milliards_usd"]     = df["pib_usd"] / 1e9
df["croissance_pop_pct"]    = (
    df["population"] / df.groupby("pays")["population"].shift(1) - 1
) * 100
df["croissance_pib_hab_pct"] = (
    df["pib_par_habitant_usd"] / df.groupby("pays")["pib_par_habitant_usd"].shift(1) - 1
) * 100

df = df.sort_values(["pays", "annee"]).reset_index(drop=True)
print(f"\nShape après nettoyage : {df.shape}")

Pays sans nom FR : 0
Pays sans sous-région : 0

=== NaN restants après imputation ===
iso3                          0
pays_en                       0
annee                         0
region_banque_mondiale        0
groupe_revenu                 0
latitude                      0
longitude                     0
population                    0
pib_usd                      24
pib_par_habitant_usd         24
croissance_pib_pct           35
inflation_pct                59
esperance_vie                 0
acces_electricite_pct         0
utilisateurs_internet_pct    33
pays                          0
sous_region                   0
dtype: int64

Note : les NaN restants concernent des pays avec données structurellement absentes (Érythrée, Soudan du Sud…) — conservés tels quels.
                     pays  annee  esperance_vie
République centrafricaine   2009         14.665
République centrafricaine   2014         40.265
République centrafricaine   2019         31.530
République centrafricaine   20

## 5. Validation & Sauvegarde (fichier propre)

In [15]:
nb_annees = ANNEE_FIN - ANNEE_DEBUT + 1
attendu   = len(AFRIQUE) * nb_annees

assert len(df) == attendu, f"{len(df)} lignes au lieu de {attendu}"
assert not df.duplicated(subset=["iso3", "annee"]).any(), "Doublons pays-année détectés"
assert (df.groupby("iso3").size() == nb_annees).all(), f"Un pays n'a pas exactement {nb_annees} années"
assert (df["population"] > 0).all(), "Population manquante ou négative"
assert df["pays"].isna().sum() == 0, "Pays sans nom FR"
assert df["sous_region"].isna().sum() == 0, "Pays sans sous-région"

calcule = df["pib_usd"] / df["population"]
ecart   = (calcule - df["pib_par_habitant_usd"]).abs() / df["pib_par_habitant_usd"]
suspects = df.loc[ecart > 0.01, "pays"].unique()
if len(suspects):
    print(f"Avertissement : écart PIB/hab > 1 % pour {len(suspects)} pays : {', '.join(sorted(suspects))}")

df.to_csv(FICHIER_PROPRE, index=False, encoding="utf-8-sig")
print(f"\nFichier propre sauvegardé : {FICHIER_PROPRE}")
print(f"Shape : {df.shape} | Doublons : {df.duplicated(subset=['iso3','annee']).sum()}")
print("\nColonnes finales :")
print(list(df.columns))

Avertissement : écart PIB/hab > 1 % pour 3 pays : Maroc, Soudan, Tanzanie

Fichier propre sauvegardé : data\dataset_economique_afrique_propre.csv
Shape : (1080, 24) | Doublons : 0

Colonnes finales :
['iso3', 'pays_en', 'annee', 'region_banque_mondiale', 'groupe_revenu', 'latitude', 'longitude', 'population', 'pib_usd', 'pib_par_habitant_usd', 'croissance_pib_pct', 'inflation_pct', 'esperance_vie', 'acces_electricite_pct', 'utilisateurs_internet_pct', 'pays', 'sous_region', 'croissance_pib_pct_outlier', 'croissance_pib_pct_ecrete', 'inflation_pct_outlier', 'inflation_pct_ecrete', 'pib_milliards_usd', 'croissance_pop_pct', 'croissance_pib_hab_pct']


## 6. Analyse exploratoire

In [16]:
colonnes_num = [
    "population", "pib_milliards_usd", "pib_par_habitant_usd",
    "croissance_pib_pct", "inflation_pct", "esperance_vie",
    "acces_electricite_pct", "utilisateurs_internet_pct",
]

print("=== Statistiques descriptives ===")
print(df[colonnes_num].agg(["min", "max", "mean", "median", "std", "skew"]).round(1).T)

print("\n=== Corrélations (Spearman) ===")
corr = df[colonnes_num].corr(method="spearman")
print(corr.round(2))

print("\n=== Top 10 PIB/hab (dernière année) ===")
derniere = df[df["annee"] == df["annee"].max()].dropna(subset=["pib_par_habitant_usd"])
cols_aff = ["pays", "sous_region", "pib_par_habitant_usd"]
print(derniere.nlargest(10,  "pib_par_habitant_usd")[cols_aff].to_string(index=False))

print("\n=== Bottom 10 PIB/hab (dernière année) ===")
print(derniere.nsmallest(10, "pib_par_habitant_usd")[cols_aff].to_string(index=False))

print("\n=== Top 5 croissance PIB ===")
print(df.nlargest(5, "croissance_pib_pct")[["pays", "annee", "croissance_pib_pct"]].to_string(index=False))

print("\n=== Bottom 8 espérance de vie ===")
print(df.nsmallest(8, "esperance_vie")[["pays", "annee", "esperance_vie"]].to_string(index=False))

print("\n=== Corrélations avec PIB/hab ===")
print(corr["pib_par_habitant_usd"].sort_values().round(2))

=== Statistiques descriptives ===
                               min          max        mean      median         std  skew
population                 82475.0  227882945.0  21877830.2  11766292.0  32155321.8   3.2
pib_milliards_usd              0.1        668.2        42.7        12.5        88.8   3.8
pib_par_habitant_usd         125.2      19141.5      2512.1      1202.3      3112.6   2.5
croissance_pib_pct           -50.3         86.8         4.0         4.3         6.1   0.6
inflation_pct                -16.9        557.2         9.4         5.3        27.8  13.2
esperance_vie                 32.5         77.2        61.3        61.1         7.0  -0.0
acces_electricite_pct          0.8        100.0        47.1        45.0        29.1   0.3
utilisateurs_internet_pct      0.0         91.0        18.9        10.9        20.5   1.3

=== Corrélations (Spearman) ===
                           population  pib_milliards_usd  pib_par_habitant_usd  croissance_pib_pct  inflation_pct  esperanc

In [17]:
med_region = (df.groupby(["sous_region", "annee"])["pib_par_habitant_usd"]
                .median().unstack("sous_region"))
print(med_region.loc[[2004, 2013, 2023]].round(0))

evol = (df.groupby("annee")[["esperance_vie", "acces_electricite_pct",
                             "utilisateurs_internet_pct"]].median().round(1))
print(evol.loc[[2004, 2013, 2023]])

derniere = df[df["annee"] == df["annee"].max()]
print(derniere[colonnes_num].corr(method="spearman")["esperance_vie"].sort_values().round(2))

sous_region  Afrique australe  Afrique centrale  Afrique de l'Est  Afrique de l'Ouest  Afrique du Nord
annee                                                                                                 
2004                   3403.0            1133.0             427.0               589.0           2500.0
2013                   5347.0            1576.0             963.0              1010.0           3875.0
2023                   4226.0            2478.0            1143.0              1288.0           3882.0
       esperance_vie  acces_electricite_pct  utilisateurs_internet_pct
annee                                                                 
2004            54.9                   29.3                        1.1
2013            61.0                   41.8                        6.6
2023            65.6                   58.2                       38.7
population                  -0.19
croissance_pib_pct          -0.02
inflation_pct                0.05
pib_milliards_usd           

In [18]:
aus = df[df["sous_region"] == "Afrique australe"]
pv = aus.pivot(index="annee", columns="pays", values="pib_par_habitant_usd")
print(pv.loc[[2013, 2023]].round(0))
print((pv.loc[2023] / pv.loc[2013] - 1).mul(100).round(0))

pays   Afrique du Sud  Botswana  Eswatini  Lesotho  Namibie
annee                                                      
2013           7332.0    6667.0    3950.0   1151.0   5347.0
2023           6034.0    7827.0    3756.0    921.0   4226.0
pays
Afrique du Sud   -18.0
Botswana          17.0
Eswatini          -5.0
Lesotho          -20.0
Namibie          -21.0
dtype: float64
